# Deriving Upper Bound on Cost for Hypervolume Metric (AddedEVScenario)

This notebook provides a step-by-step guide to calculating the **maximum daily cost** for the **AddedEVScenario**. This scenario builds upon the ConstantPricesScenario by including the charging of an electric vehicle (EV). The primary data sources from the `./commonpower/finetuning/data/` directory are:

* `ICLR_load.csv` – baseline building load (MW)
* `ICLR_pv.csv` – PV generation (MW)
* `ToU_prices.csv` – grid buying price (€/MWh)
* `ICLR_load_with_ev.csv` – EV charging power (kW)

The result of this notebook will serve as the **lower bound on the reward function** (because reward = −cost) for the PCN approach when computing the hypervolume metric.

### Imports, File Paths, and CSV Parsing

In [36]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path().resolve().parent / 'finetuning/data'

LOAD_FILE = BASE_DIR / 'ICLR_load.csv'
PV_FILE = BASE_DIR / 'ICLR_pv.csv'
PRICE_FILE = BASE_DIR / 'ToU_prices.csv'
EV_FILE = BASE_DIR / 'ICLR_load_with_ev.csv'

#####################################################################

load = pd.read_csv(LOAD_FILE, parse_dates=['t']).rename(columns={'p':'load_MW'})
pv   = pd.read_csv(PV_FILE,   parse_dates=['t']).rename(columns={'p':'pv_MW'})
prices = (pd.read_csv(PRICE_FILE, parse_dates=['t'])
            .rename(columns={'psib_mwh':'price_EUR_per_MWh'}))
ev = pd.read_csv(EV_FILE, parse_dates=['t']).rename(columns={'diff':'load_ev_MW'})

for df, name in [(load,'Load'), (pv,'PV'), (prices,'Prices'), (ev, 'EV (Load)')]:
    print(f"{name:12}: {df.shape[0]:5d} rows, time span {df['t'].min()} → {df['t'].max()}")

Load        :  8784 rows, time span 2016-01-01 00:00:00 → 2016-12-31 23:00:00
PV          :  8784 rows, time span 2016-01-01 00:00:00 → 2016-12-31 23:00:00
Prices      :  8784 rows, time span 2016-01-01 00:00:00 → 2016-12-31 23:00:00
EV (Load)   :  8784 rows, time span 2016-01-01 00:00:00 → 2016-12-31 23:00:00


# Case 1: Lower Bound without a Heat Pump

### 1. Net Load Calculation (MW)
For each 1-hour interval, we calculate the net power that is either imported from or exported to the grid. In the `AddedEVScenario`, this includes the power consumed by the EV charger.

$\text{Net Load}_{t}\;[\text{MW}] \;=\; (\text{Load}_{t}\;[\text{MW}] + \text{EV Load}_{t}\;[\text{MW}]) \;-\; \text{PV}_{t}\;[\text{MW}]$

### 2. Interval Cost Calculation (Euro)

To determine the upper-bound cost, we make the following worst-case assumptions:

- We **pay** for all imported power (positive net load).
- We **do not receive any revenue** for exported power (negative net load is treated as zero).

The cost for each interval is calculated as follows:

$$
\text{Cost}_{t} \;=\; 
\max\!\bigl(0,\text{Net Load}_{t}\bigr)
\;\cdot\;
\text{Price}_{t} \;[\text{Euro}/ \text{MWh}]
$$

### 3. Daily Cost Calculation (Euro)

The total daily cost is the sum of the costs of all 24 hourly intervals in a day:

$$
\text{Daily Cost}_{d} 
\;=\;
\sum_{t\in d}\text{Cost}_{t}
$$

### 4. Maximum Daily Cost

Then we identify the highest daily cost throughout the entire year.

## 1. Compute Net Load and Preprocessing

In [37]:
df = (load.merge(pv, on='t', how='inner')
      .merge(prices[['t','price_EUR_per_MWh']], on='t')
      .merge(ev[['t', 'load_ev_MW']], on='t', how='inner'))

# Convert MW -> kW so that multiplying by hours = kWh
df['load_kW'] = df['load_MW'] * 1_000
df['pv_kW']   = df['pv_MW']   * 1_000
df['load_ev_kW'] = df['load_ev_MW'] * 1_000

# Calculate net load including EV charging load
df['net_load_kW'] = df['load_kW'] + df['load_ev_kW'] - df['pv_kW']
df['net_load_MW'] = df['load_MW'] + df['load_ev_MW'] - df['pv_MW']

#print(df[df['t'].dt.date == pd.to_datetime('2016-12-06').date()])

## 2. Worst-Case Interval Cost

In [40]:
df['price_EUR_per_kWh'] = df['price_EUR_per_MWh'] / 1_000
df['interval_cost_EUR'] = df['net_load_kW'].clip(lower=0) * df['price_EUR_per_kWh']

print("Interval cost statistics (EUR):")
print(df['interval_cost_EUR'].describe(percentiles=[0.9, 0.99]))

Interval cost statistics (EUR):
count    8784.000000
mean        5.836754
std        12.771145
min        -4.637646
50%         1.516848
90%        13.689350
99%        65.898133
max       177.502132
Name: interval_cost_EUR, dtype: float64


## 3. Aggregate to Daily Cost

In [41]:
df['date'] = df['t'].dt.date
daily_cost = df.groupby('date')['interval_cost_EUR'].sum().sort_values(ascending=False)

print(f"Max daily cost: {daily_cost.iloc[0]:.2f} € on {daily_cost.index[0]}")

daily_cost.head()

Max daily cost: 846.79 € on 2016-01-20


date
2016-01-20    846.790757
2016-11-08    666.558697
2016-11-10    575.997114
2016-12-02    555.930823
2016-12-12    545.876705
Name: interval_cost_EUR, dtype: float64

# Case 2: Lower Bound with a Heat Pump

Now we are adding the electrical consumption of the described heat pump. The result gives a **lower bound on reward** (upper bound on cost + penalty) again for hyper‑volume metrics. Again we need the Data sources as above:

* `ICLR_load.csv` – baseline building load (MW)
* `ICLR_pv.csv` – PV generation (MW)
* `ToU_prices.csv` – grid buying price (€/MWh)
* `ICLR_load_with_ev.csv` – EV charging power (kW)

For the heat pump case we also need to consider:
* `DE_Temperature_and_COP2016.csv` – outside temperature and heat‑pump COP (hourly)


### 1. Simplified Heat-Pump Electrical Power Cost

To simplify the calculation, we use a fixed maximum power consumption for the heat pump and multiply it by the cost of electricity over the maximum time period.

$$ \text{Heat Pump Cost} = P_{\text{hp,max}} \cdot \text{Price} \cdot \text{Time} $$

Where:
* $P_{\text{hp,max}}$ = 5 kW (the maximum power of the heat pump as defined in `scenarios.py`)
* Price is the electricity price in €/kWh
* Time is the duration in hours


In [42]:
P_HP_MAX = 5.0  # kW
EPISODE_H = 24  # hours

# Calculate additional cost from heat pump
df['heat_pump_cost_EUR'] = P_HP_MAX * df['price_EUR_per_kWh'] * EPISODE_H

# Add hp cost to daily cost
daily_cost_with_hp = daily_cost + df.groupby(df['t'].dt.date)['heat_pump_cost_EUR'].sum()

# Find worst case daily cost
worst_cost_with_hp = daily_cost_with_hp.max()
worst_day_with_hp = daily_cost_with_hp.idxmax()

print(f"Max daily cost with heat pump: €{worst_cost_with_hp:,.2f} on {worst_day_with_hp}")

Max daily cost with heat pump: €985.99 on 2016-01-20
